# 04 · Evaluation & Deployment Readiness  (CRISP-DM phases 5–6)

Phase 5 – evaluate the chosen model against the business criterion on the untouched test set (using the **train-only** model, because the deployed model has seen the test months).
Phase 6 – produce the next-month forecast for every series exactly as the API will, and smoke-test the serving library.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
import prep
FIG = os.path.join(ROOT, "reports", "figures"); os.makedirs(FIG, exist_ok=True)
RAW = os.path.join(ROOT, "data", "wfp_food_prices_zmb.csv")
PROC = os.path.join(ROOT, "data", "processed"); os.makedirs(PROC, exist_ok=True)
MODELS = os.path.join(ROOT, "models"); os.makedirs(MODELS, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.autolayout": True})
def save(name):
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show()

import joblib
from serve_lib import Forecaster
meta = json.load(open(os.path.join(MODELS, "metadata.json"))); print(json.dumps(meta, indent=1))
F = pd.read_csv(os.path.join(PROC, "features.csv"), parse_dates=["month","next_month"]).sort_values(["series_id","month"]).reset_index(drop=True)
F["rk"] = F.groupby("series_id").cumcount(); F["n"] = F.groupby("series_id").month.transform("size")
F["split"] = np.where(F.rk >= F.n*0.85, "test", np.where(F.rk >= F.n*0.70, "val", "train"))
B = joblib.load(os.path.join(MODELS, "best_model_trainonly.joblib"))
te = F[(F.split=="test")].copy()
yh = B["model"].predict(te[B["features"]])
sig = te.roll_std_6.fillna(te.roll_std_3).fillna(B["sigma_default"]).clip(lower=B["sigma_floor"]).values
te["pred"] = te.price*np.exp(yh); te["lo"] = te.price*np.exp(yh-B["conformal_q"]*sig); te["hi"] = te.price*np.exp(yh+B["conformal_q"]*sig)
te = te[te.y_imputed==0].copy()
te["ape"] = (te.pred-te.y_price).abs()/te.y_price*100; te["naive_ape"] = (te.price-te.y_price).abs()/te.y_price*100
print(f"TEST  model MAPE {te.ape.mean():.2f}% | naive MAPE {te.naive_ape.mean():.2f}% | MAE {np.abs(te.pred-te.y_price).mean():.3f} ZMW | MdAPE {te.ape.median():.2f}%")

## 5.1 Business-criterion check (MAPE < 10 %)

In [ ]:
verdict = "MEETS" if te.ape.mean() < 10 else "DOES NOT MEET"
print(f"Test MAPE = {te.ape.mean():.2f}%  ->  {verdict} the 10% target;  validation MAPE = {meta['validation_MAPE_pct']:.2f}%")
print(f"share of test forecasts within 10% of actual: {(te.ape<10).mean()*100:.1f}%")

## 5.2 Where is the model good / bad?

In [ ]:
bc = te.groupby("commodity").agg(MAPE=("ape","mean"), naive_MAPE=("naive_ape","mean"), n=("ape","size")).sort_values("MAPE")
fig, ax = plt.subplots(figsize=(8,6)); y = np.arange(len(bc))
ax.barh(y-.2, bc.MAPE, .4, label="model", color="#2a6f97"); ax.barh(y+.2, bc.naive_MAPE, .4, label="naive", color="#cfcfcf")
ax.axvline(10, color="k", ls="--", lw=1); ax.set_yticks(y); ax.set_yticklabels(bc.index, fontsize=8); ax.legend(); ax.set_xlabel("test MAPE (%)")
ax.set_title("Test MAPE by commodity"); save("04_mape_by_commodity.png"); display(bc.round(2))

In [ ]:
bp = te.groupby("province").agg(MAPE=("ape","mean"), naive_MAPE=("naive_ape","mean"), n=("ape","size")).sort_values("MAPE")
fig, ax = plt.subplots(figsize=(7,3.6)); bp[["MAPE","naive_MAPE"]].plot.bar(ax=ax, color=["#2a6f97","#cfcfcf"])
ax.axhline(10, color="k", ls="--", lw=1); ax.set_ylabel("test MAPE (%)"); ax.set_title("Test MAPE by province"); plt.xticks(rotation=30, ha="right")
save("04_mape_by_province.png"); display(bp.round(2))

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(14,4))
ax[0].scatter(te.y_price, te.pred, s=5, alpha=.35, color="#2a6f97"); lim=[te.y_price.min(), te.y_price.max()]
ax[0].plot(lim, lim, "k--", lw=1); ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted"); ax[0].set_title("Actual vs predicted (log axes)")
ax[1].hist(np.log(te.y_price/te.pred).clip(-.5,.5), bins=60, color="#8fb339"); ax[1].set_title("Residual log(actual/predicted)")
te["yr"] = te.next_month.dt.year; te.groupby("yr").ape.mean().plot.bar(ax=ax[2], color="#e07a5f"); ax[2].axhline(10, color="k", ls="--", lw=1); ax[2].set_title("Test MAPE by forecast year")
save("04_diagnostics.png")

## 5.3 Prediction-interval quality

In [ ]:
te["covered"] = (te.y_price>=te.lo)&(te.y_price<=te.hi)
print(f"90% interval coverage on test: {te.covered.mean()*100:.1f}%  | mean width {((te.hi-te.lo)/te.pred).mean()*100:.1f}% of forecast")
cov = te.groupby("commodity").covered.mean().mul(100).sort_values()
fig, ax = plt.subplots(figsize=(7,5)); cov.plot.barh(ax=ax, color="#2a6f97"); ax.axvline(90, color="k", ls="--", lw=1)
ax.set_title("Interval coverage by commodity (%) – dashed = 90% nominal"); save("04_interval_coverage.png")

## 5.4 Sensitivity to imputed data
How do errors change when the *current* month (the model's starting point) is itself imputed?

In [ ]:
te_all = F[F.split=="test"].copy(); te_all["pred"]=te_all.price*np.exp(B["model"].predict(te_all[B["features"]]))
te_all = te_all[te_all.y_imputed==0]; te_all["ape"]=(te_all.pred-te_all.y_price).abs()/te_all.y_price*100
print(te_all.groupby("is_imputed_t").ape.agg(["mean","median","size"]).round(2).rename(index={0:"start month observed",1:"start month imputed"}))

## 6.1 Next-month forecasts for every series (exactly as served)
`Forecaster` is the same class the API uses: forecast month = last observation month + 1.

In [ ]:
fc = Forecaster(os.path.join(MODELS, "best_model.joblib"), os.path.join(PROC, "panel_monthly.csv"))
allf = pd.DataFrame(fc.predict_rows(fc.last)); allf.to_csv(os.path.join(ROOT,"reports","next_month_forecasts.csv"), index=False)
print(len(allf), "series forecast"); display(allf.head(10))
print(fc.predict_rows(fc.resolve("fresh beans","Central")[1]))

In [ ]:
# plot history + forecast + interval for a few series
pick = ["Central|Beans|KG","Lusaka|Maize meal (white, roller)|KG","Copperbelt|Sugar|KG","Southern|Eggs|30 pcs"]
pick = [p for p in pick if p in fc.last.index][:4]
fig, axes = plt.subplots(2,2, figsize=(12,6))
for ax, sid in zip(axes.ravel(), pick):
    g = fc.panel[fc.panel.series_id==sid].tail(36); r = allf[allf.series_id==sid].iloc[0]
    ax.plot(g.month, g.price, color="#2a6f97", marker="o", ms=3); ax.scatter(g.month[g.is_imputed==1], g.price[g.is_imputed==1], color="red", marker="x", s=40, zorder=3)
    d = pd.to_datetime(r.forecast_date+"-01"); ax.errorbar([d],[r.predicted_price], yerr=[[r.predicted_price-r.lower_bound],[r.upper_bound-r.predicted_price]], fmt="s", color="#e07a5f", capsize=4)
    ax.set_title(sid, fontsize=9)
save("04_example_forecasts.png")

## 6.2 Conclusions & deployment notes
* Model, features, conformal quantile and codes are stored in `models/best_model.joblib`; the API loads it together with `data/processed/panel_monthly.csv` (history incl. the `is_imputed` flag).
* Re-training: re-run notebooks 02 → 03 with new WFP data; the API picks up the new artefacts at restart.
* Limitations: monthly prices of this dataset are noisy (market sample changes between rounds); several commodities end years ago, so a forecast is still "next month after the last record" but may be stale – the API flags this.